# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hicham1236/Intern-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row represents a single web page (URL) at a specific snapshot in time.

**Time Window:** For this setup, we are using the provided anonymized snapshot from the starter dataset.

In [1]:
import os, sys, subprocess
import pandas as pd

# Setup environment to load data
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load data and verify grain
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Total rows in dataset: {len(df)}")
print("Grain check: The dataset is aggregated at the page level (one row = one page metrics snapshot).")

Total rows in dataset: 30000
Grain check: The dataset is aggregated at the page level (one row = one page metrics snapshot).


## 2. Fields: feature / label / context / excluded

**Features:** content_age_days, days_since_last_update, impressions_90d, avg_position, ctr. (These are available before the decision is made).

**Label:** is_declining (We will derive this proxy from trend_direction == 'down').

**Context:** Any ID or baseline metrics kept only for reporting, not for training.

**Excluded:**

**trend_pct:** Excluded because it is a direct leak of the label.

**Pages with impressions_90d == 0:** Excluded because dead pages offer no useful decay signal for the model.

In [2]:
# Create the label
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Define buckets
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
label = 'is_declining'
excluded_leakage = ['trend_pct', 'trend_direction']

print(f"Features mapped: {features}")
print(f"Label mapped: '{label}'")
print(f"Excluded from training to prevent leakage: {excluded_leakage}")

Features mapped: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
Label mapped: 'is_declining'
Excluded from training to prevent leakage: ['trend_pct', 'trend_direction']


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we verify the integrity of our selected slice. We will check for missing values in our feature columns and apply our exclusion rule (filtering out pages with 0 impressions) to see how many valid rows survive.

In [3]:
print("--- Missing Values Check ---")
print(df[features].isna().sum())

print("\n--- Availability Check (Excluding 0 impressions) ---")
# Create boolean mask
df['is_active'] = df['impressions_90d'] > 0
# Query where it IS TRUE
df_valid = df.query('is_active == True').copy()

print(f"Original row count: {len(df)}")
print(f"Valid rows surviving the filter: {len(df_valid)}")
print(f"Excluded {len(df) - len(df_valid)} zero-traffic rows.")

--- Missing Values Check ---
content_age_days          0
days_since_last_update    0
impressions_90d           0
avg_position              0
ctr                       0
dtype: int64

--- Availability Check (Excluding 0 impressions) ---
Original row count: 30000
Valid rows surviving the filter: 30000
Excluded 0 zero-traffic rows.


## 4. Data limits

**Limitation:** This dataset relies entirely on historical search metrics (impressions, positions, CTR) and basic age metadata. It contains no semantic text data, readability scores, or keyword mapping.

**Impact:** The model can accurately flag that a page is declining, but it cannot diagnose why the content is failing or tell the writer what specific paragraphs need rewriting. It is a prioritization tool, not a content generation tool.

In [4]:
# Displaying the columns to prove the absence of semantic/text features
print("Available columns in the dataset:")
for col in df_valid.columns:
    print(f"- {col}")

print("\nConclusion: No HTML, text, or semantic quality features exist in this slice.")

Available columns in the dataset:
- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct
- is_declining
- is_active

Conclusion: No HTML, text, or semantic quality features exist in this slice.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.